# ResNet50-V2

##  Step 1: Import Necessary libraries

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

## Step 2: CHANGE ONLY THIS SECTION

In [7]:
IMAGE_SIZE = (224, 224)          # <- Change input size
BATCH_SIZE = 32
EPOCHS = 10

TRAIN_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\train_data"
VAL_DIR = r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\val_data"
TEST_DIR=r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data"
NUM_CLASSES = 5                  # <- Number of classes

BASE_MODEL_NAME ="ResNet50V2"    # <- Options: VGG16, ResNet50, MobileNetV2, etc

FREEZE_LAYERS = True             # <- Freeze base model
FINE_TUNE_AT = None              # <- Set layer index to unfreeze later

## Step 3: Data Preparation || Data Pipeline

In [8]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   horizontal_flip=True,
                                   zoom_range=0.2)

val_datagen = ImageDataGenerator(rescale=1./255)
test_generator = train_datagen.flow_from_directory(TEST_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical',
                                               shuffle=False)
train_data = train_datagen.flow_from_directory(TRAIN_DIR,
                                               target_size=IMAGE_SIZE,
                                               batch_size=BATCH_SIZE,
                                               class_mode='categorical')

val_data = val_datagen.flow_from_directory(VAL_DIR,
                                           target_size=IMAGE_SIZE,
                                           batch_size=BATCH_SIZE,
                                           class_mode='categorical')

Found 15 images belonging to 5 classes.
Found 50 images belonging to 5 classes.
Found 9 images belonging to 5 classes.


## Step 4: Model Building: Load PreTrained Model

In [9]:
def get_base_model(name):
    if name == "VGG16":
        return tf.keras.applications.VGG16(weights='imagenet',
                                           include_top=False,
                                           input_shape=(*IMAGE_SIZE, 3))
    elif name == "ResNet50":
        return tf.keras.applications.ResNet50(weights='imagenet',
                                              include_top=False,
                                              input_shape=(*IMAGE_SIZE, 3))
    elif name == "MobileNetV2":
        return tf.keras.applications.MobileNetV2(weights='imagenet',
                                                 include_top=False,
                                                 input_shape=(*IMAGE_SIZE, 3))
    elif name == "ResNet50V2":
        return tf.keras.applications.ResNet50V2(weights='imagenet',
                                                 include_top=False,
                                                 input_shape=(*IMAGE_SIZE, 3))

base_model = get_base_model(BASE_MODEL_NAME)

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step


### ====================================================================

# The Most Important 2 Steps in Transfer Learning: 
## 1. FREEZE BASE MODEL

In [10]:
if FREEZE_LAYERS:
    for layer in base_model.layers:
        layer.trainable = False

## 2. ADD CUSTOM HEAD

In [11]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_preact_bn        │ (None, 56, 56, 64)        │             256 │ pool1_pool[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_preact_relu      │ (None, 56, 56, 64)        │               0 │ conv2_block1_preact_bn[0]… │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,096 │ conv2_block1_preact_relu[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_pad            │ (None, 58, 58, 64)        │               0 │ conv2_block1_1_relu[0][0]  │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,864 │ conv2_block1_2_pad[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │               

 Total params: 24,090,629 (91.90 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,564,800 (89.89 MB)

### ====================================================================

### Step 4: Model Building Continues..It's COMPILATION Time.

In [12]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## Step 5: Model Training | Model Evaluation | Model Testing

In [13]:
history = model.fit(train_data,
                    validation_data=val_data,
                    epochs=EPOCHS)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.1400 - loss: 2.3889 - val_accuracy: 0.2222 - val_loss: 1.9971
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 753ms/step - accuracy: 0.1800 - loss: 2.1408 - val_accuracy: 0.2222 - val_loss: 1.9029
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 783ms/step - accuracy: 0.1400 - loss: 2.2359 - val_accuracy: 0.1111 - val_loss: 1.8467
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 756ms/step - accuracy: 0.2400 - loss: 1.6334 - val_accuracy: 0.1111 - val_loss: 1.8173
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.3400 - loss: 1.8910 - val_accuracy: 0.1111 - val_loss: 1.7986
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 759ms/step - accuracy: 0.2800 - loss: 1.7553 - val_accuracy: 0.1111 - val_loss: 1.7888
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 755ms/step - accuracy: 0.2200 - loss: 1.6884 - val_accuracy: 0.1111 - val_loss: 1.7808
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 1s/step - accuracy: 0.3000 - loss: 1.5579 - val_accuracy: 0.1111 - val_loss: 1.7722
E

##  (OPTIONAL Step): FINE-TUNING with new weights(NOT SUGGESTED)

In [14]:
if FINE_TUNE_AT is not None:
    for layer in base_model.layers[FINE_TUNE_AT:]:
        layer.trainable = True

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print("Starting Fine-Tuning...")

    history_fine = model.fit(
        train_data,
        validation_data=val_data,
        epochs=5
    )

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_preact_bn        │ (None, 56, 56, 64)        │             256 │ pool1_pool[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_preact_relu      │ (None, 56, 56, 64)        │               0 │ conv2_block1_preact_bn[0]… │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,096 │ conv2_block1_preact_relu[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_pad            │ (None, 58, 58, 64)        │               0 │ conv2_block1_1_relu[0][0]  │
│ (ZeroPadding2D)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,864 │ conv2_block1_2_pad[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │               

 Total params: 25,142,289 (95.91 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,564,800 (89.89 MB)

 Optimizer params: 1,051,660 (4.01 MB)

## Export the Intelligence File.

### Question: How to use this file for Prediction?

In [15]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\Vijay\images (4).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step   
Predicted Person: Dhruv


In [16]:
from tensorflow.keras.preprocessing import image
import numpy as np

# Load the image
img = image.load_img( r"C:\Users\Abdul\Deep Learning\08_CNN\Image Classification\test_data\DQ\images (12).jpg", target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict
prediction = model.predict(img_array)

# Get predicted class index
predicted_index = np.argmax(prediction)

# Convert class index to person name
class_names = {v: k for k, v in test_generator.class_indices.items()}

# Print the person's name
print("Predicted Person:", class_names[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Predicted Person: Ajith


# THE END!